# dataset_v6_spatial811 冒烟测试

用途：自动识别两个 Kaggle 账号中的 v6 数据，严格核对版本后，以少量 batch 验证数据读取、前向、反向、验证、测试和权重保存链路。

运行前：打开 GPU；打开 Internet（首次克隆代码、安装缺失依赖及下载 ImageNet 编码器权重需要）。冒烟测试指标不是正式实验结果。

In [ ]:
import os, sys, subprocess, importlib.util

print('Python:', sys.version)
print('Kaggle:', os.path.isdir('/kaggle'))
subprocess.run(['nvidia-smi'], check=False)

missing = []
for module_name, package_name in [('rasterio', 'rasterio'), ('segmentation_models_pytorch', 'segmentation-models-pytorch')]:
    if importlib.util.find_spec(module_name) is None:
        missing.append(package_name)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('依赖检查完成')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REPO_DIR = Path('/kaggle/working/lunar-linear-v6-smoke')

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'目录已存在但不是 Git 仓库，请换一个 REPO_DIR: {REPO_DIR}')
else:
    print('仓库已存在，本次不重复克隆:', REPO_DIR)

PROJECT_DIR = REPO_DIR / 'LTL-Net'
assert PROJECT_DIR.is_dir(), PROJECT_DIR
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('branch:', REPO_BRANCH)
print('commit:', commit)
print('project:', PROJECT_DIR)

In [ ]:
import json
import numpy as np
import rasterio

DATA_CANDIDATES = [
    Path('/kaggle/input/datasets/yuanssy/v6data/dataset_v6_spatial811'),
    Path('/kaggle/input/datasets/changyasong/v6data/dataset_v6_spatial811'),
]
EXPECTED_TILES = {'train': 494, 'val': 63, 'test': 55}
EXPECTED_MEAN = np.array([0.14634284944967138, 0.6073079654785911, 0.19560334930002798, 0.5146424734578662, 0.4709976669495292])
EXPECTED_STD = np.array([0.06626422267714538, 0.34714253416282376, 0.2144817435820576, 0.15443829274986642, 0.15999780505000355])

def tif_files(folder):
    return sorted([*folder.glob('*.tif'), *folder.glob('*.tiff')])

def validate_dataset(root):
    if not root.is_dir():
        return False, f'目录不存在: {root}'
    summary_path = root / 'dataset_summary.json'
    stats_path = root / 'normalization_stats.json'
    if not summary_path.is_file() or not stats_path.is_file():
        return False, '缺少 dataset_summary.json 或 normalization_stats.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    stats = json.loads(stats_path.read_text(encoding='utf-8'))
    for split, expected in EXPECTED_TILES.items():
        declared = summary.get(split, {}).get('tiles')
        images = tif_files(root / split / 'image')
        masks = tif_files(root / split / 'mask')
        if declared != expected or len(images) != expected or len(masks) != expected:
            return False, f'{split}: summary/image/mask={declared}/{len(images)}/{len(masks)}, expected={expected}'
        if {p.stem for p in images} != {p.stem for p in masks}:
            return False, f'{split}: image-mask 文件名不一一对应'
    mean = np.asarray(stats.get('mean'), dtype=float)
    std = np.asarray(stats.get('std'), dtype=float)
    if mean.shape != (5,) or std.shape != (5,) or not np.allclose(mean, EXPECTED_MEAN, atol=1e-12) or not np.allclose(std, EXPECTED_STD, atol=1e-12):
        return False, f'归一化统计不匹配: mean={mean.tolist()}, std={std.tolist()}'
    return True, '通过'

valid_roots = []
for candidate in DATA_CANDIDATES:
    ok, message = validate_dataset(candidate)
    print(('PASS' if ok else 'SKIP'), candidate, '-', message)
    if ok:
        valid_roots.append(candidate)
assert valid_roots, '两个指定路径均未通过版本核验；请确认 Kaggle Dataset 已挂载且版本正确'
DATA_ROOT = valid_roots[0]
print('使用数据:', DATA_ROOT)

In [ ]:
# 逐 split 抽取首、中、末三对影像/掩膜，检查尺寸、通道、有限值和标签范围。
for split in EXPECTED_TILES:
    images = tif_files(DATA_ROOT / split / 'image')
    masks = tif_files(DATA_ROOT / split / 'mask')
    mask_by_stem = {p.stem: p for p in masks}
    indices = sorted({0, len(images) // 2, len(images) - 1})
    for index in indices:
        image_path = images[index]
        mask_path = mask_by_stem[image_path.stem]
        with rasterio.open(image_path) as src:
            image = src.read(masked=True)
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
        assert image.shape == (5, 512, 512), (image_path, image.shape)
        assert mask.shape == (512, 512), (mask_path, mask.shape)
        assert set(np.unique(mask).tolist()) <= {0, 1, 2, 3, 4}, (mask_path, np.unique(mask))
        valid_values = image.compressed()
        assert np.isfinite(valid_values).all(), image_path
    print(f'{split}: 抽检 {len(indices)} 对通过')
print('数据结构与样本读取检查全部通过')

## 短跑配置

默认只跑 DeepLabV3+：2 个 epoch，每个 train/val/test 阶段最多 5 个 batch。若要测试 LTL-Net，把 `MODEL_KIND` 改成 `ltl` 后重新运行本格及后续格。正式训练必须把 `MAX_STEPS` 改为 0，并采用完整训练轮数。

In [ ]:
MODEL_KIND = 'baseline'  # 'baseline' 或 'ltl'
BASELINE_MODEL = 'DeepLabV3Plus'
ENCODER = 'resnet50'
SEED = 42
EPOCHS = 2
MAX_STEPS = 5

assert MODEL_KIND in {'baseline', 'ltl'}
assert MAX_STEPS > 0, '这个 notebook 用于短跑；正式训练请另建运行并将 max_steps 设为 0'
run_name = f'v6_smoke_{BASELINE_MODEL if MODEL_KIND == "baseline" else "LTLNet"}_seed{SEED}'
if MODEL_KIND == 'baseline':
    command = [sys.executable, str(PROJECT_DIR / 'scripts' / 'train_baseline.py'),
               '--model', BASELINE_MODEL, '--encoder', ENCODER]
else:
    command = [sys.executable, str(PROJECT_DIR / 'scripts' / 'train_ltl.py'),
               '--encoder', ENCODER]
command += ['--data-dir', str(DATA_ROOT), '--seed', str(SEED), '--epochs', str(EPOCHS),
            '--max-steps', str(MAX_STEPS), '--run-name', run_name]
print('运行命令:')
print(' '.join(command))

In [ ]:
subprocess.check_call(command, cwd=PROJECT_DIR)
print('冒烟训练完成')

In [ ]:
RESULT_DIR = Path('/kaggle/working') / f'result_{run_name}'
METRICS_PATH = RESULT_DIR / 'metrics.json'
CHECKPOINT_PATH = RESULT_DIR / 'best_model.pth'
assert METRICS_PATH.is_file(), METRICS_PATH
assert CHECKPOINT_PATH.is_file(), CHECKPOINT_PATH
result = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
assert result['max_steps'] == MAX_STEPS
assert Path(result['data_root']) == DATA_ROOT
print(json.dumps(result, ensure_ascii=False, indent=2))
print('\n冒烟测试通过。输出目录:', RESULT_DIR)
print('checkpoint:', CHECKPOINT_PATH)
print('注意：该 checkpoint 只用于确认流程，不用于论文结果或后续推理。')